## *1. Librerias y Dependencias*

In [1]:
import chromadb
import sentence_transformers
import pypdf

/home/victor/miniconda3/envs/mongo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## _2. Cargar PDF_

In [2]:
from pypdf import PdfReader

reader = PdfReader("Apuntes_Ampliacion_BBDD.pdf")

In [3]:
pdf_data = ""
for page in reader.pages:
    pdf_data += page.extract_text()

In [4]:
print(pdf_data)

Víctor Manuel Peiró Martínez 
FIIS 3ºA  Ampliación de Bases de Datos 
APUNTES 
 
   
1 
Contenido 
Introducción .................................................................................................................................. 4 
Tipos de Datos ........................................................................................................................... 4 
BBDD Relacionales .................................................................................................................. 4 
Desajuste de Impedancia ........................................................................................................... 5 
La Caída del Imperio Relacional ............................................................................................... 5 
Clústeres ................................................................................................................................ 5 
NoSQL ..............................................................

## _3. Chunking_

In [5]:
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    if overlap >= chunk_size:
        return []

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap

    return chunks

In [6]:
chunks = create_chunks(pdf_data)

In [7]:
for idx,chunk in enumerate(chunks):
    print(f"Chunk {idx}")
    print(chunk)

Chunk 0
Víctor Manuel Peiró Martínez 
FIIS 3ºA  Ampliación de Bases de Datos 
APUNTES 
 
   
1 
Contenido 
Introducción .................................................................................................................................. 4 
Tipos de Datos ........................................................................................................................... 4 
BBDD Relacionales ..............................................................................................
Chunk 1
nales .................................................................................................................. 4 
Desajuste de Impedancia ........................................................................................................... 5 
La Caída del Imperio Relacional ............................................................................................... 5 
Clústeres ...................................................................................

## _4. Establecimiento de Cliente y Embeddings_

In [8]:
client = chromadb.PersistentClient(path="./chroma_data")
col_rag = client.get_or_create_collection(name="rag_collection", metadata={"hnsw:space":"cosine"})

In [9]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer("all-MiniLM-l6-v2")

In [10]:
embs = emb_model.encode(chunks).tolist()

col_rag.add(
    documents=chunks,
    embeddings=embs,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

In [11]:
col_rag.count()

119

## _5. Consultas_

In [14]:
query = "Queries aggregadas de MongoDB"

query_embedding = emb_model.encode([query]).tolist()

results = col_rag.query(
    query_embeddings=query_embedding,
    n_results=3
)


In [15]:
for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Resultado {i+1} ---\n")
    print(doc[:500])


--- Resultado 1 ---

con $jsonSchema 
Documentacion: https://www.mongodb.com   
Rendimiento 
Se usan índices en variables utilizadas en filtros 
Evitar el uso de operaciones join y anidar la información mientras no sea excesiva 
Usar indexado inverso cuando se realicen consultas sobre texto 
Posibles Problemas de Diseño: 
• Muchas colecciones 
• Array de muchos elementos 
• Mucha información en un solo documento 
• Muchos índices 
Índices 
Tipos de índice: 
• Ascendente (1) o Descendiente (-1) 
• De texto (text) 
• 

--- Resultado 2 ---

 Agregado: Se permite agrupar/anidar datos en un mismo documento. 
Se accede a los documentos por claves siendo estas identificadores o variables indexadas 
Ejemplo: MongoDB 
  
7 
Orientadas a Clave-Valor 
Solo existe un valor para una clave. 
Modelo Agregado: El valor puede tener todo tipo de datos: Variables, listas, sets, tablas 
hash, arrays asociativos, etc… 
Ejemplo: Redis 
Orientadas a Columnas 
Datos almacenados en tuplas con nombre único, da